# DWN sm on SENA binary MNIST split (GPU required)

1. Runtime → Change runtime type → **GPU**
2. Upload `data/X_train.npy`, `y_train.npy`, `X_test.npy`, `y_test.npy` (from `Revision_V13/baseline_dwn/data`)
3. Run all cells
4. Download `results/dwn_sm_sena_train.json` and `checkpoints/*.pt`

In [ ]:
!pip -q install torch-dwn
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime first'
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files
import os
os.makedirs('data', exist_ok=True)
print('Upload the four .npy files from baseline_dwn/data')
uploaded = files.upload()
for name in uploaded:
    open(f'data/{name}', 'wb').write(uploaded[name])
    print('saved', name)

In [ ]:
# Paste / adapt train_sena_dwn_sm.py here, or upload the script and run:
# !python train_sena_dwn_sm.py --seeds 0,1,2,3,4
from pathlib import Path
import json, time, numpy as np, torch
from torch import nn
from torch.nn.functional import cross_entropy
import torch_dwn as dwn

DATA, OUT, CKPT = Path('data'), Path('results'), Path('checkpoints')
OUT.mkdir(exist_ok=True); CKPT.mkdir(exist_ok=True)

def load_split():
    return (torch.from_numpy(np.load(DATA/'X_train.npy')),
            torch.from_numpy(np.load(DATA/'y_train.npy')),
            torch.from_numpy(np.load(DATA/'X_test.npy')),
            torch.from_numpy(np.load(DATA/'y_test.npy')))

def build_model(n_in):
    return nn.Sequential(
        dwn.LUTLayer(n_in, 1000, n=6, mapping='learnable'),
        dwn.LUTLayer(1000, 500, n=6),
        dwn.GroupSum(10, tau=1/0.245))

def lr_for_epoch(e):
    return [1e-2,1e-3,1e-4,1e-5][min(e//30, 3)]

@torch.no_grad()
def accuracy(model, x, y, batch=512):
    model.eval(); preds=[]
    for i in range(0,len(x),batch):
        preds.append(model(x[i:i+batch].cuda()).argmax(1).cpu())
    return float((torch.cat(preds)==y).float().mean())

def train_one(seed, epochs=100, batch_size=128):
    torch.manual_seed(seed); np.random.seed(seed)
    xtr,ytr,xte,yte = load_split()
    model = build_model(xtr.size(1)).cuda()
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)
    hist=[]; t0=time.perf_counter(); n=len(xtr)
    for epoch in range(epochs):
        lr=lr_for_epoch(epoch)
        for g in opt.param_groups: g['lr']=lr
        model.train(); perm=torch.randperm(n); correct=total=0; last=0.0
        for i in range(0,n,batch_size):
            idx=perm[i:i+batch_size]
            bx,by=xtr[idx].cuda(), ytr[idx].cuda()
            opt.zero_grad(set_to_none=True)
            out=model(bx); loss=cross_entropy(out,by); loss.backward(); opt.step()
            last=float(loss.item()); correct+=int((out.argmax(1)==by).sum()); total+=by.size(0)
        row=dict(epoch=epoch+1, lr=lr, loss=last, train_acc=correct/total, test_acc=accuracy(model,xte,yte))
        hist.append(row)
        print(f'seed={seed} ep={epoch+1}/{epochs} test={row["test_acc"]:.4f}', flush=True)
    path=CKPT/f'dwn_sm_sena_seed{seed}.pt'
    torch.save(dict(seed=seed, state_dict=model.state_dict(), n_in=int(xtr.size(1)),
                    test_acc=hist[-1]['test_acc'], best_test_acc=max(h['test_acc'] for h in hist),
                    history=hist, elapsed_s=time.perf_counter()-t0), path)
    return dict(seed=seed, test_acc=hist[-1]['test_acc'], best_test_acc=max(h['test_acc'] for h in hist),
                checkpoint=str(path))

runs=[train_one(s) for s in range(5)]
acc=np.array([r['test_acc'] for r in runs])
summary=dict(runs=runs, test_acc_mean=float(acc.mean()), test_acc_std=float(acc.std(ddof=1)))
(OUT/'dwn_sm_sena_train.json').write_text(json.dumps(summary, indent=2))
print(summary['test_acc_mean'], '±', summary['test_acc_std'])

In [ ]:
from google.colab import files
import shutil
shutil.make_archive('dwn_sena_out', 'zip', root_dir='.', base_dir='.')
# Prefer a lean zip of results+checkpoints only:
shutil.make_archive('dwn_sena_ckpts', 'zip', root_dir='.', base_dir='checkpoints')
files.download('results/dwn_sm_sena_train.json')
files.download('dwn_sena_ckpts.zip')